# LSTM Experiments — ESC-50 Environmental Sound Classification

**Purpose.** Full 9-architecture experiment grid and 5-fold promotion for the LSTM investigation. Assumes `LSTM baseline.ipynb` already passed validation on both the `short` and `long` segment variants.

**Stages** (see `LSTM Experiments.md` for full rationale):

| Cell | Stage | What it does | Compute |
|---|---|---|---|
| Step 13 | Stage 1 screening | 9 architectures × `VARIANT_LIST` on fold 1 | 9-18 runs |
| Step 14 | Stage 1b promotion | winner + baseline × 5 folds | 10 runs |
| Step 15 | Phase 2 (optional) | winner short architecture on `long`, fold 1 | 1 long run |
| Step 16 | Stage 2a (optional) | all 9 architectures on `long`, fold 1 | 9 long runs |
| Step 17 | Stage 2b (optional) | long winner × 5 folds | 5 long runs |

**Run-name convention.** The segment variant is **always** suffixed onto the run name (e.g. `LSTM_Run1_Baseline_short`, `LSTM_Run3_CapacityUp_long`) so artifacts never collide across variants. 5-fold runs add `_fold{N}`.

**Default `VARIANT_LIST = ['short']` in Step 13.** The long variant is reached via the opt-in Stage 2 cells (15-17) so compute is controlled. To run both variants up-front in Stage 1, set `VARIANT_LIST = ['short', 'long']` in Step 13.

Shared data pipeline (Steps 1–7) is identical to the Piczak baseline notebook. LSTM-specific code begins at **Step 8**.

In [ ]:
# Step 1: Imports and configuration

%pip install -q librosa

import os
import time
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

SEED = 99
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ZIP_PATH    = '/content/drive/MyDrive/ESC Project Deep Learning 5888 Data Dump/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/ESC Project Deep Learning 5888 Data Dump/LSTM'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH  = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')
    wav_files = [f for f in files if f.endswith('.wav')]
    if wav_files and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)
print('Audio folder: ', AUDIO_DIR)
print('Output folder:', OUTPUT_DIR)
print('Device:       ', DEVICE)

In [ ]:
# Step 2: Shared spectrogram hyperparameters
# These match the Piczak baseline exactly — do not change.

SAMPLE_RATE = 22050
N_FFT       = 1024
HOP_LENGTH  = 512
N_MELS      = 60
NUM_CLASSES = 50

VARIANT_CONFIGS = {
    'short': {
        'segment_frames':     41,
        'segment_hop_frames': 20,
    },
    'long': {
        'segment_frames':     101,
        'segment_hop_frames': 10,
    },
}

In [ ]:
# Step 3: Load ESC-50 metadata

df = pd.read_csv(CSV_PATH)
display(df.head())

In [ ]:
# Step 4: Feature extraction — log-mel spectrogram + delta

def extract_features(audio_path, sr=SAMPLE_RATE, n_fft=N_FFT,
                     hop_length=HOP_LENGTH, n_mels=N_MELS):
    """Load one ESC-50 clip and return a (2, N_MELS, T) array:
       channel 0 = log-mel spectrogram
       channel 1 = delta
    """
    y, _ = librosa.load(audio_path, sr=sr)
    y    = librosa.util.normalize(y)

    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
        n_mels=n_mels, power=2.0
    )
    log_mel = librosa.power_to_db(mel_spec)
    delta   = librosa.feature.delta(log_mel)

    return np.stack([log_mel, delta], axis=0).astype(np.float32)

In [ ]:
# Step 5: Segment extraction

def extract_segments(features, segment_frames, segment_hop_frames):
    """Extract overlapping spectrogram segments; discard silent ones."""
    n_frames = features.shape[2]
    segments = []
    for start in range(0, n_frames - segment_frames + 1, segment_hop_frames):
        segment = features[:, :, start:start + segment_frames]
        if np.mean(np.abs(segment[0])) > 1e-6:
            segments.append(segment.astype(np.float32))
    return segments

In [ ]:
# Step 6: Build segment samples and Dataset

def build_segment_samples(dataframe, audio_dir, segment_frames, segment_hop_frames):
    samples = []
    for idx in range(len(dataframe)):
        row        = dataframe.iloc[idx]
        audio_path = os.path.join(audio_dir, row['filename'])
        label      = int(row['target'])
        filename   = row['filename']

        features = extract_features(audio_path)
        segments = extract_segments(features, segment_frames, segment_hop_frames)
        for segment in segments:
            samples.append((segment, label, filename))

        if (idx + 1) % 200 == 0:
            print(f'  Processed {idx + 1}/{len(dataframe)} clips...')

    print(f'  Total segments: {len(samples)} from {len(dataframe)} clips')
    return samples


class ESC50Dataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        segment, label, filename = self.samples[idx]
        return torch.from_numpy(segment).float(), label, filename

In [ ]:
# Step 7: Train/test dataloader factory

def get_train_test_loaders(dataframe, audio_dir, test_fold,
                           variant_name='short', batch_size=64, num_workers=2):
    """Returns (train_loader, test_loader) for one fold."""
    train_df = dataframe[dataframe['fold'] != test_fold].reset_index(drop=True)
    test_df  = dataframe[dataframe['fold'] == test_fold].reset_index(drop=True)

    seg_frames = VARIANT_CONFIGS[variant_name]['segment_frames']
    seg_hop    = VARIANT_CONFIGS[variant_name]['segment_hop_frames']

    train_samples = build_segment_samples(train_df, audio_dir, seg_frames, seg_hop)
    test_samples  = build_segment_samples(test_df,  audio_dir, seg_frames, seg_hop)

    train_loader = DataLoader(ESC50Dataset(train_samples), batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(ESC50Dataset(test_samples),  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

---
## LSTM Model

All LSTM-specific code is below. The only thing that changes between experimental runs is `LSTM_CONFIG`.

In [ ]:
# Step 8: LSTM experiment config
#
# This is the single dict to edit between runs.
# Architectural factors varied across the 9-run experiment plan:
#   hidden_size  : 64 | 128 | 256
#   num_layers   : 1  | 2   | 3
#   bidirectional: True | False
#   pooling      : 'final' | 'mean'
#
# dropout and learning_rate are held fixed across all runs —
# they are training hyperparameters, not architectural ones.
# Varying them would confound architectural comparisons.
# Exception: if a run produces NaN loss, halve lr to 5e-4
# as a debugging step before concluding the architecture is the problem.

LSTM_CONFIG = {
    # --- identity ---
    'model_name':    'LSTM_Run1_Baseline',
    'variant_name':  'short',
    'save_dir':      OUTPUT_DIR,

    # --- architectural factors (vary these between runs) ---
    'hidden_size':   128,
    'num_layers':    2,
    'bidirectional': True,
    'pooling':       'final',   # 'final' or 'mean'

    # --- training hyperparameters (fixed across all runs) ---
    'dropout':       0.3,
    'learning_rate': 1e-3,
    'num_epochs':    60,
    'batch_size':    64,
}

In [ ]:
# Step 9: LSTM model — config-driven
#
# Input  : (B, 2, 60, T)  — batch, channels, mel_bins, time_frames
# Reshape: (B, T, 120)    — treat each time frame as one sequence step
# Output : (B, num_classes)
#
# Pooling options
#   'final' — concatenate last-layer forward + backward hidden states.
#             Relies on the LSTM to carry relevant info to the last step.
#   'mean'  — average all timestep outputs.
#             More robust when the key sound event occurs mid-clip.

class LSTMBaseline(nn.Module):
    def __init__(self, config):
        super().__init__()
        input_size         = 2 * N_MELS                  # 2 channels * 60 mel bins = 120
        hidden_size        = config['hidden_size']
        num_layers         = config['num_layers']
        dropout            = config['dropout']
        bidirectional      = config['bidirectional']
        self.pooling       = config['pooling']
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * self.num_directions, NUM_CLASSES)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # Forget gate bias = 1.0: encourages remembering across
                # long spans early in training (standard LSTM trick)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(1.0)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        B, C, M, T = x.shape
        x = x.view(B, C * M, T).permute(0, 2, 1)          # (B, T, 120)
        output, (h_n, _) = self.lstm(x)

        if self.pooling == 'mean':
            h = output.mean(dim=1)                          # (B, hidden * directions)
        else:  # 'final'
            if self.bidirectional:
                h = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (B, hidden * 2)
            else:
                h = h_n[-1]                                 # (B, hidden)

        return self.classifier(self.dropout(h))


def build_lstm_from_config(config, device=DEVICE):
    """Instantiate and move model to device directly from config dict."""
    return LSTMBaseline(config).to(device)


# Quick sanity check
model_test = build_lstm_from_config(LSTM_CONFIG)
dummy      = torch.randn(2, 2, N_MELS, VARIANT_CONFIGS['short']['segment_frames']).to(DEVICE)
print('Input shape: ', dummy.shape)
print('Output shape:', model_test(dummy).shape)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'Parameters:  {total_params:,}')
del model_test, dummy

In [ ]:
# Step 10: Training loop
#
# Key implementation notes:
#   - Gradient clipping (max_norm=1.0) guards against exploding gradients,
#     which caused the NaN losses in the Piczak 'long' variant.
#   - Cosine LR decay reduces the learning rate smoothly over training.
#   - Two test metrics tracked per epoch:
#       test_acc       — segment-level accuracy (every segment graded vs.
#                        its clip label independently). Matches the raw
#                        numbers the project notebooks have been reporting.
#       clip_test_acc  — clip-level accuracy via probability voting:
#                        average softmax over all segments of a clip,
#                        then argmax. Matches Piczak's SP/LP evaluation
#                        (paper §3.2, "taking into account the probabilities
#                        predicted for each segment"), which is the number
#                        that's directly comparable to Piczak's 44% B
#                        baseline and ~60-65% CNN results on ESC-50.
#   - best_* / best_epoch track peak generalization rather than the
#     final-epoch value.

def _aggregate_clip_probs(probs, labels_cpu, fnames, store):
    """Accumulate per-clip softmax sums for probability voting."""
    for fn, prob, lab in zip(fnames, probs, labels_cpu):
        if fn not in store['sum']:
            store['sum'][fn] = np.zeros(prob.shape[0], dtype=np.float64)
            store['label'][fn] = int(lab)
        store['sum'][fn] += prob


def _clip_accuracy(store):
    """Resolve accumulated clip probabilities to clip-level accuracy."""
    if not store['sum']:
        return 0.0
    correct = 0
    for fn, prob_sum in store['sum'].items():
        if int(np.argmax(prob_sum)) == store['label'][fn]:
            correct += 1
    return correct / len(store['sum'])


def run_model_lstm(model, train_loader, test_loader, config, device=DEVICE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['num_epochs']
    )

    history = {
        'train_loss':     [],
        'train_acc':      [],
        'test_acc':       [],   # segment-level
        'clip_test_acc':  [],   # clip-level (probability voting)
    }
    best_test_acc      = 0.0
    best_epoch         = 0
    best_clip_test_acc = 0.0
    best_clip_epoch    = 0
    start_time         = time.time()

    for epoch in range(config['num_epochs']):
        # --- train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for xb, yb, _ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            correct      += (logits.argmax(1) == yb).sum().item()
            total        += yb.size(0)

        scheduler.step()
        train_loss = running_loss / total
        train_acc  = correct / total

        # --- evaluate (segment-level AND clip-level via probability voting) ---
        model.eval()
        seg_correct, seg_total = 0, 0
        clip_store = {'sum': {}, 'label': {}}

        with torch.no_grad():
            for xb, yb, fnames in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                probs  = torch.softmax(logits, dim=1).cpu().numpy()

                seg_correct += (logits.argmax(1) == yb).sum().item()
                seg_total   += yb.size(0)

                _aggregate_clip_probs(probs, yb.cpu().numpy(), fnames, clip_store)

        test_acc      = seg_correct / seg_total
        clip_test_acc = _clip_accuracy(clip_store)

        # --- track best (separate peaks for segment- and clip-level) ---
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch    = epoch + 1
        if clip_test_acc > best_clip_test_acc:
            best_clip_test_acc = clip_test_acc
            best_clip_epoch    = epoch + 1

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['clip_test_acc'].append(clip_test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f'[{config["model_name"]}] Epoch {epoch+1:03d}/{config["num_epochs"]} | '
                f'loss {train_loss:.4f} | train {train_acc:.4f} | '
                f'seg {test_acc:.4f} | clip {clip_test_acc:.4f} | '
                f'best clip {best_clip_test_acc:.4f} (ep {best_clip_epoch})'
            )

    results = {
        'model':                 config['model_name'],
        'hidden_size':           config['hidden_size'],
        'num_layers':            config['num_layers'],
        'bidirectional':         config['bidirectional'],
        'pooling':               config['pooling'],
        'final_train_loss':      history['train_loss'][-1],
        'final_train_acc':       history['train_acc'][-1],
        'final_test_acc':        history['test_acc'][-1],        # segment-level
        'final_clip_test_acc':   history['clip_test_acc'][-1],   # clip-level
        'best_test_acc':         best_test_acc,                  # segment-level peak
        'best_epoch':            best_epoch,
        'best_clip_test_acc':    best_clip_test_acc,             # clip-level peak (paper-comparable)
        'best_clip_epoch':       best_clip_epoch,
        'training_time_secs':    round(time.time() - start_time, 1),
    }
    return history, results

In [ ]:
# Step 11: Results saving (matches shared team convention: save_dir/model_name)

def save_results(history, results, config):
    save_path = Path(config['save_dir']) / config['model_name']
    save_path.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(save_path / 'history.csv', index=False)
    pd.DataFrame([results]).to_csv(save_path / 'results.csv', index=False)

    with open(save_path / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)

    plt.figure(figsize=(10, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs, history['train_loss'],    label='Train Loss')
    plt.plot(epochs, history['train_acc'],     label='Train Acc')
    plt.plot(epochs, history['test_acc'],      label='Test Acc (segment)')
    if 'clip_test_acc' in history:
        plt.plot(epochs, history['clip_test_acc'], label='Test Acc (clip, prob-vote)',
                 linewidth=2)
    best_clip_ep = results.get('best_clip_epoch', results.get('best_epoch'))
    plt.axvline(x=best_clip_ep, color='gray', linestyle='--',
                label=f'Best clip epoch ({best_clip_ep})')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.title(config['model_name'])
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / 'curves.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f'Saved to: {save_path}')

In [ ]:
# Step 12: Single-run wrapper
#
# Usage: edit LSTM_CONFIG above, then call run_experiment().
# Everything else cascades from the config dict.

def run_experiment(config, dataframe, audio_dir, test_fold, device=DEVICE):
    """Build loaders -> instantiate model -> train -> save. One call per experiment row."""
    print(f'\n{"="*60}')
    print(f'  {config["model_name"]}  |  fold {test_fold}')
    print(f'  hidden={config["hidden_size"]}  layers={config["num_layers"]}  '
          f'bidir={config["bidirectional"]}  pooling={config["pooling"]}')
    print(f'{"="*60}')

    train_loader, test_loader = get_train_test_loaders(
        dataframe=dataframe,
        audio_dir=audio_dir,
        test_fold=test_fold,
        variant_name=config['variant_name'],
        batch_size=config['batch_size'],
    )
    model = build_lstm_from_config(config, device)
    history, results = run_model_lstm(model, train_loader, test_loader, config, device)
    results['fold'] = test_fold
    save_results(history, results, config)
    return history, results

In [ ]:
# Step 13: Stage 1 screening — 9 architectures × VARIANT_LIST on fold 1
#
# Each row is validated on fold 1 first. Step 14 then picks the winner
# (by clip-level accuracy) and promotes it to 5-fold for the final
# reported numbers.
#
# Architectural factors only — dropout and lr are fixed across all runs.
# See `LSTM Experiments.md` for full rationale and the expected signal
# for each run.
#
# VARIANT_LIST controls which segment-length variants are screened in
# this Stage 1 cell:
#   * Default ['short']           → 9 runs, fold 1. Use the optional
#                                    Stage 2 cells (Steps 15-17) to explore
#                                    the long variant afterwards.
#   * ['short', 'long']           → 18 runs on fold 1. Stages 2a/2b
#                                    become redundant and will auto-skip.
#
# Run name convention: `<base_model_name>_<variant>` (variant always
# included) so artifacts never collide across short / long.

VARIANT_LIST = ['short']  # set to ['short', 'long'] to sweep both segment lengths

EXPERIMENT_GRID = [
    # name                          hidden  layers  bidir   pooling
    ('LSTM_Run1_Baseline',           128,   2,      True,  'final'),
    ('LSTM_Run2_CapacityDown',        64,   2,      True,  'final'),
    ('LSTM_Run3_CapacityUp',         256,   2,      True,  'final'),
    ('LSTM_Run4_DepthDown',          128,   1,      True,  'final'),
    ('LSTM_Run5_DepthUp',            128,   3,      True,  'final'),
    ('LSTM_Run6_Unidirectional',     128,   2,      False, 'final'),
    ('LSTM_Run7_MeanPooling',        128,   2,      True,  'mean'),
    ('LSTM_Run8_CapacityMeanPool',   256,   1,      True,  'mean'),
    ('LSTM_Run9_UniMeanPool',        128,   2,      False, 'mean'),
]

all_results = []

for variant in VARIANT_LIST:
    for base_model_name, hidden, layers, bidir, pooling in EXPERIMENT_GRID:
        # Variant is always part of the run name so short/long artifacts never
        # collide and the variant is unambiguous in every downstream CSV/plot.
        run_name = f'{base_model_name}_{variant}'
        config = {
            **LSTM_CONFIG,                # inherit shared training hyperparameters
            'model_name':    run_name,
            'variant_name':  variant,
            'hidden_size':   hidden,
            'num_layers':    layers,
            'bidirectional': bidir,
            'pooling':       pooling,
        }
        _, results = run_experiment(config, df, AUDIO_DIR, test_fold=1)
        results['variant'] = variant
        results['base_model_name'] = base_model_name
        all_results.append(results)

# Summary across all runs (may span variants).
# Sorted by clip-level accuracy (probability voting) — this is the
# paper-comparable metric against Piczak's ~60-65% CNN numbers on ESC-50.
# Segment-level columns are kept for debugging / training-curve continuity.
summary = pd.DataFrame(all_results)[[
    'model', 'variant', 'base_model_name',
    'hidden_size', 'num_layers', 'bidirectional', 'pooling',
    'best_clip_test_acc', 'best_clip_epoch', 'final_clip_test_acc',
    'best_test_acc', 'best_epoch', 'final_test_acc',
    'training_time_secs'
]].sort_values('best_clip_test_acc', ascending=False)

print(f'\n--- {len(all_results)}-Run Summary (fold 1, ranked by clip-level acc) ---')
display(summary)

In [ ]:
# Step 14: Promotion to 5-fold cross-validation
#
# Take the winner from the fold-1 screening grid (and the baseline Run 1)
# and evaluate each across all 5 folds. Report mean ± std test accuracy —
# these are the headline numbers to compare against:
#   * Piczak's ESC-50 random-forest baseline "B"   = 44%     (clip-level)
#   * Piczak's best CNN variant (LP)                = 64.5%  (clip-level)
#
# Winner is picked by CLIP-level accuracy (probability voting) because
# that's the metric directly comparable to the paper's numbers.
#
# If the winner IS the baseline, only the baseline is run 5-fold.
# Otherwise both are run so you have an ablation pair.

arch_by_name = {tup[0]: tup for tup in EXPERIMENT_GRID}

winner_row = summary.sort_values('best_clip_test_acc', ascending=False).iloc[0]
winner_base = winner_row['base_model_name']
winner_variant = winner_row['variant']

baseline_base = 'LSTM_Run1_Baseline'
baseline_variant = VARIANT_LIST[0]

promote_specs = [(winner_base, winner_variant)]
if winner_base != baseline_base or winner_variant != baseline_variant:
    promote_specs.append((baseline_base, baseline_variant))

print('Promoting to 5-fold:')
for base_name, variant in promote_specs:
    print(f'  {base_name} on variant={variant}')

fold_records = []
for base_name, variant in promote_specs:
    _, hidden, layers, bidir, pooling = arch_by_name[base_name]
    for test_fold in range(1, 6):
        config = {
            **LSTM_CONFIG,
            'model_name':    f'{base_name}_{variant}_fold{test_fold}',
            'variant_name':  variant,
            'hidden_size':   hidden,
            'num_layers':    layers,
            'bidirectional': bidir,
            'pooling':       pooling,
        }
        _, results = run_experiment(config, df, AUDIO_DIR, test_fold=test_fold)
        fold_records.append({
            'architecture':        base_name,
            'variant':             variant,
            'fold':                test_fold,
            'best_clip_test_acc':  results['best_clip_test_acc'],
            'final_clip_test_acc': results['final_clip_test_acc'],
            'best_clip_epoch':     results['best_clip_epoch'],
            'best_test_acc':       results['best_test_acc'],
            'final_test_acc':      results['final_test_acc'],
            'best_epoch':          results['best_epoch'],
        })

fold_df = pd.DataFrame(fold_records)

summary_5fold = (
    fold_df
    .groupby(['architecture', 'variant'])[[
        'best_clip_test_acc', 'final_clip_test_acc',
        'best_test_acc', 'final_test_acc',
    ]]
    .agg(['mean', 'std'])
    .round(4)
)

print('\n--- 5-Fold Results (clip-level is the paper-comparable number) ---')
display(summary_5fold)

print('\nReference points for ESC-50 (clip-level):')
print('  Piczak random-forest baseline (B):      0.4400')
print('  Piczak best CNN variant (LP):           0.6450')

summary_path = Path(LSTM_CONFIG['save_dir']) / 'lstm_5fold_summary.csv'
fold_df.to_csv(summary_path, index=False)
print(f'\n5-fold records saved to: {summary_path}')

In [ ]:
# Step 15: Phase 2 — Long-variant extension (winning architecture only)
#
# Takes the winning architecture from the fold-1 screening and trains it
# on the 'long' segment variant (101 frames, ~2.35s) for fold 1 only.
# Purpose: test whether the LSTM's recurrent inductive bias pays off on
# longer sequences, where the Piczak CNN's training recipe collapsed (2%).
#
# This is a targeted extension, not part of the main reported result.
# If the long fold-1 number meaningfully beats the short fold-1 number
# for the same architecture, consider promoting it to full 5-fold on long.
#
# Skipped automatically if 'long' was already in VARIANT_LIST.

if 'long' in VARIANT_LIST:
    print("'long' already included in the main screening grid — skipping Phase 2.")
else:
    _, hidden, layers, bidir, pooling = arch_by_name[winner_base]

    long_config = {
        **LSTM_CONFIG,
        'model_name':    f'{winner_base}_long_extension',
        'variant_name':  'long',
        'hidden_size':   hidden,
        'num_layers':    layers,
        'bidirectional': bidir,
        'pooling':       pooling,
    }

    print(f'Running Phase 2: {winner_base} on long variant, fold 1')
    print(f'  hidden={hidden}  layers={layers}  bidir={bidir}  pooling={pooling}')

    _, long_results = run_experiment(long_config, df, AUDIO_DIR, test_fold=1)

    # Retrieve the same architecture's short fold-1 numbers from the screening grid
    short_match = summary[
        (summary['base_model_name'] == winner_base) &
        (summary['variant'] == 'short')
    ]
    if len(short_match) > 0:
        short_fold1_clip = short_match.iloc[0]['best_clip_test_acc']
        short_fold1_seg  = short_match.iloc[0]['best_test_acc']
    else:
        short_fold1_clip = None
        short_fold1_seg  = None

    phase2_comparison = pd.DataFrame([
        {
            'architecture':                 winner_base,
            'variant':                      'short',
            'best_clip_test_acc_fold1':     short_fold1_clip,
            'best_test_acc_fold1':          short_fold1_seg,
            'source':                       'screening (Cell 14)',
        },
        {
            'architecture':                 winner_base,
            'variant':                      'long',
            'best_clip_test_acc_fold1':     long_results['best_clip_test_acc'],
            'best_test_acc_fold1':          long_results['best_test_acc'],
            'source':                       'Phase 2 extension',
        },
    ])

    print(f'\n--- Short vs Long (fold 1, winning architecture) ---')
    display(phase2_comparison)

    if short_fold1_clip is not None:
        delta_clip = long_results['best_clip_test_acc'] - short_fold1_clip
        sign = '+' if delta_clip >= 0 else ''
        print(f'\nDelta clip-level (long - short): {sign}{delta_clip:.4f}')
        if delta_clip > 0.02:
            print('  Long meaningfully outperforms short on fold 1 (clip-level).')
            print('  Consistent with Piczak §3.3: "longer time scales ... slight improvements".')
            print('  Recommend promoting this architecture to 5-fold on the long variant.')
        elif delta_clip < -0.02:
            print('  Long underperforms short. Longer context is not helping this architecture.')
            print('  (Inconsistent with Piczak — could indicate overfitting on the smaller')
            print('   effective training set from 90% overlap, or insufficient epochs.)')
        else:
            print('  Long and short are roughly comparable — within noise for a single fold.')

    phase2_path = Path(LSTM_CONFIG['save_dir']) / 'lstm_phase2_long_extension.csv'
    phase2_comparison.to_csv(phase2_path, index=False)
    print(f'\nPhase 2 comparison saved to: {phase2_path}')

In [ ]:
# Step 16: Stage 2a (optional) — Full LONG-variant screening (9 runs × fold 1)
#
# Parallels Stage 1's short-variant screening (Cell 14) but on the long
# segment variant (101 frames, ~2.3 s, 90% overlap). Piczak §3.3 reports
# long gives slight improvements over short on ESC-50, so running the
# full architectural matrix on long is the proper way to reproduce his
# SM/SP vs LM/LP comparison.
#
# WHEN TO RUN THIS:
#   * Stage 1 is complete and the Phase 2 extension (Cell 16) suggested
#     long is competitive with short, AND you have compute to spare.
#   * You want a faithful reproduction of Piczak's full experimental
#     matrix (all architectures × both segment variants).
#
# COMPUTE COST:
#   * Each long run is ~3.3x a short run (2.5x frames × 1.3x segments/clip).
#   * 9 long runs ≈ 30 short-equivalent runs.
#   * On a T4 Colab: roughly 2-6 hours depending on utilisation.
#
# This cell skips automatically if 'long' was already in Stage 1's
# VARIANT_LIST (in that case the data is already in `summary`).

if 'long' in VARIANT_LIST:
    print("'long' already covered by Stage 1's VARIANT_LIST — skipping Stage 2a.")
    print("Use the rows of `summary` where variant == 'long' directly.")
else:
    long_results = []

    for base_model_name, hidden, layers, bidir, pooling in EXPERIMENT_GRID:
        config = {
            **LSTM_CONFIG,
            'model_name':    f'{base_model_name}_long',
            'variant_name':  'long',
            'hidden_size':   hidden,
            'num_layers':    layers,
            'bidirectional': bidir,
            'pooling':       pooling,
        }
        _, results = run_experiment(config, df, AUDIO_DIR, test_fold=1)
        results['variant'] = 'long'
        results['base_model_name'] = base_model_name
        long_results.append(results)

    long_summary = pd.DataFrame(long_results)[[
        'model', 'variant', 'base_model_name',
        'hidden_size', 'num_layers', 'bidirectional', 'pooling',
        'best_clip_test_acc', 'best_clip_epoch', 'final_clip_test_acc',
        'best_test_acc', 'best_epoch', 'final_test_acc',
        'training_time_secs'
    ]].sort_values('best_clip_test_acc', ascending=False)

    print(f'\n--- {len(long_results)}-Run LONG Summary (fold 1, ranked by clip-level) ---')
    display(long_summary)

    long_screen_path = Path(LSTM_CONFIG['save_dir']) / 'lstm_long_screening_summary.csv'
    long_summary.to_csv(long_screen_path, index=False)
    print(f'\nLong screening saved to: {long_screen_path}')

In [ ]:
# Step 17: Stage 2b (optional) — 5-fold promotion of the LONG-variant winner
#
# Parallels Stage 1's 5-fold promotion (Cell 15) but for the long variant.
# Produces the mean ± std clip-level accuracy that is directly comparable
# to Piczak's LM/LP reported numbers (~64.5% best, ~62-65% typical).
#
# REQUIRES:
#   * Either Stage 2a (Cell 17) has completed → uses `long_summary`, OR
#   * Long was in Stage 1's VARIANT_LIST → pulls long rows from `summary`.
#
# COMPUTE COST: 5 long folds × 1 architecture ≈ 16 short-equivalent runs.
# On a T4 Colab: roughly 1-3 hours.

proceed = True
long_winner_row = None

if 'long' in VARIANT_LIST:
    long_slice = summary[summary['variant'] == 'long']
    if len(long_slice) == 0:
        print("No long rows found in Stage 1 `summary` — cannot promote.")
        proceed = False
    else:
        long_winner_row = (
            long_slice.sort_values('best_clip_test_acc', ascending=False).iloc[0]
        )
        print('Using long winner from Stage 1 `summary`.')
elif 'long_summary' in globals():
    long_winner_row = long_summary.iloc[0]
    print('Using long winner from Stage 2a `long_summary`.')
else:
    print('Neither Stage 2a (Cell 17) nor a long-inclusive Stage 1 has run.')
    print('Run Cell 17 first (or set VARIANT_LIST = [\'short\', \'long\'] in Cell 14).')
    proceed = False

if proceed:
    long_winner_base = long_winner_row['base_model_name']
    _arch_by_name = {tup[0]: tup for tup in EXPERIMENT_GRID}
    _, hidden, layers, bidir, pooling = _arch_by_name[long_winner_base]

    print(f'\nPromoting long winner to 5-fold: {long_winner_base}')
    print(f'  hidden={hidden}  layers={layers}  bidir={bidir}  pooling={pooling}')
    print(f'  fold-1 best clip-level: {long_winner_row["best_clip_test_acc"]:.4f}')

    long_fold_records = []
    for test_fold in range(1, 6):
        config = {
            **LSTM_CONFIG,
            'model_name':    f'{long_winner_base}_long_fold{test_fold}',
            'variant_name':  'long',
            'hidden_size':   hidden,
            'num_layers':    layers,
            'bidirectional': bidir,
            'pooling':       pooling,
        }
        _, results = run_experiment(config, df, AUDIO_DIR, test_fold=test_fold)
        long_fold_records.append({
            'architecture':        long_winner_base,
            'variant':             'long',
            'fold':                test_fold,
            'best_clip_test_acc':  results['best_clip_test_acc'],
            'final_clip_test_acc': results['final_clip_test_acc'],
            'best_clip_epoch':     results['best_clip_epoch'],
            'best_test_acc':       results['best_test_acc'],
            'final_test_acc':      results['final_test_acc'],
            'best_epoch':          results['best_epoch'],
        })

    long_fold_df = pd.DataFrame(long_fold_records)
    long_summary_5fold = (
        long_fold_df
        .groupby(['architecture', 'variant'])[[
            'best_clip_test_acc', 'final_clip_test_acc',
            'best_test_acc', 'final_test_acc',
        ]]
        .agg(['mean', 'std'])
        .round(4)
    )

    print('\n--- Long 5-Fold Results (clip-level is paper-comparable) ---')
    display(long_summary_5fold)

    print('\nReference points for ESC-50 (clip-level, from Piczak 2015):')
    print('  Random-forest baseline (B):          0.4400')
    print('  CNN SM / SP (short, 41 frames):      ~0.58-0.62')
    print('  CNN LM / LP (long, 101 frames):       0.6450')

    long_5fold_path = Path(LSTM_CONFIG['save_dir']) / 'lstm_long_5fold_summary.csv'
    long_fold_df.to_csv(long_5fold_path, index=False)
    print(f'\nLong 5-fold records saved to: {long_5fold_path}')